In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("github_token")

In [2]:
!git clone https://{token}@github.com/seprow/pixcell.git

Cloning into 'pixcell'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 182 (delta 20), reused 182 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (182/182), 57.43 KiB | 3.38 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [4]:
from pathlib import Path

import sys
sys.path.append('/kaggle/working/pixcell/src')

import SimpleITK as sitk
sitk.ProcessObject_SetGlobalWarningDisplay(False)

from pixcell.configs import Config
from pixcell.utils import load_yaml


YAML_PATH = Path("/kaggle/input/datasets/seprow/config1/config.yaml") 

config = Config.from_dict(
        load_yaml(YAML_PATH)
    )

In [5]:
from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    EnsureTyped,
    NormalizeIntensityd,
    RandScaleIntensityd,
    RandShiftIntensityd,
    DivisiblePadd,
    AsDiscreted
)

from pixcell.utils.registry import TRANSFORM_PIPELINE

from pixcell.data.engine.transforms import(
    Windowingd
)

@TRANSFORM_PIPELINE.register("train_transform")
def train_transform():

    return Compose([

        EnsureChannelFirstd(keys=["image", 'ich_multiclass_segmentation_series'], channel_dim="no_channel"),
        EnsureTyped(keys=["image", 'ich_multiclass_segmentation_series']),
        Windowingd(keys="image", window_center=40 ,window_width=80),
        
        AsDiscreted(
            keys='ich_multiclass_segmentation_series',
            to_onehot=6,
        ),
        DivisiblePadd(
            keys=["image", 'ich_multiclass_segmentation_series'],
            k=8,
            mode=("constant", "minimum"),
            method="end"

        ),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
        RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),

    ])

@TRANSFORM_PIPELINE.register("val_transform")
def val_transform():

    return Compose([
        EnsureChannelFirstd(keys=["image", 'ich_multiclass_segmentation_series'], channel_dim="no_channel"),
        EnsureTyped(keys=["image", 'ich_multiclass_segmentation_series']),
        Windowingd(keys="image", window_center=40 ,window_width=80),
        
        AsDiscreted(
            keys='ich_multiclass_segmentation_series',
            to_onehot=6,
        ),
        DivisiblePadd(
            keys=["image", 'ich_multiclass_segmentation_series'],
            k=8,
            mode=("constant", "minimum"),
            method="end"

        ),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        
    ])

In [6]:
from monai.networks.nets import SegResNet

from pixcell.models import BaseModel
from pixcell.utils.registry import MODEL_REGISTRY


@MODEL_REGISTRY.register("model")
class SegResNetModel(BaseModel):

    def __init__(
        self,
        in_channels=1,
        out_channels=6,
        blocks_down=[1,2,2],
        blocks_up=[1,1],
        init_filters=8,
        dropout_prob=0.2,
    ):
        super().__init__()

        self.model = SegResNet(
            blocks_down=blocks_down,
            blocks_up=blocks_up,
            init_filters=init_filters,
            in_channels=in_channels,
            out_channels=out_channels,
            dropout_prob=dropout_prob,
        )

    def forward(self, x):

            logits = self.model(x)

            return {
                'ich_multiclass_segmentation_series': logits
            }



In [ ]:
from pixcell.runners import SupervisedRunner

runner = SupervisedRunner(config).run()

2026-08-05 19:08:31 | INFO     | pixcell.runners.supervised_runner:__init__:44 - Initializing runner...

========== Fold 1 ==========
2026-08-05 19:12:02 | INFO     | pixcell.training.trainer.supervised:fit:308 - Training started
2026-08-05 19:12:02 | INFO     | pixcell.training.trainer.supervised:fit:315 - ========== Epoch 1 ==========
2026-08-05 19:12:02 | INFO     | pixcell.training.trainer.supervised:train_epoch:90 - Train epoch 1 | batches=161


Train 1: 100%|██████████| 161/161 [06:27<00:00,  2.40s/it, loss=0.994]

2026-08-05 19:18:29 | INFO     | pixcell.training.trainer.supervised:validate_epoch:186 - Validation epoch 1 | batches=37



Val 1: 100%|██████████| 37/37 [00:36<00:00,  1.01it/s, loss=0.998]

2026-08-05 19:19:06 | INFO     | pixcell.training.trainer.supervised:fit:330 - Epoch 1 | {'train/ich_multiclass_segmentation_series/dice': 0.99594528593632, 'train/total': 0.99594528593632, 'val/ich_multiclass_segmentation_series/dice': [0.014802816323935986, 0.0318407267332077, 0.01994801126420498, 0.010970816016197205, 0.0020796414464712143], 'val/total': 0.9980865249762664}
2026-08-05 19:19:06 | INFO     | pixcell.training.trainer.supervised:save_checkpoint:257 - Saved checkpoint -> checkpoints/best_model.pt
2026-08-05 19:19:06 | INFO     | pixcell.training.trainer.supervised:fit:315 - ========== Epoch 2 ==========
2026-08-05 19:19:06 | INFO     | pixcell.training.trainer.supervised:train_epoch:90 - Train epoch 2 | batches=161


...


Train 35: 100%|██████████| 161/161 [06:25<00:00,  2.39s/it, loss=0.894]

2026-08-05 23:15:22 | INFO     | pixcell.training.trainer.supervised:validate_epoch:186 - Validation epoch 35 | batches=37



Val 35: 100%|██████████| 37/37 [00:33<00:00,  1.12it/s, loss=0.954]

2026-08-05 23:15:55 | INFO     | pixcell.training.trainer.supervised:fit:330 - Epoch 35 | {'train/ich_multiclass_segmentation_series/dice': 0.9324745913470014, 'train/total': 0.9324745913470014, 'val/ich_multiclass_segmentation_series/dice': [0.0018274434842169285, 0.5939681529998779, 0.1750643402338028, 0.0025134303141385317, 4.087750494363718e-05], 'val/total': 0.9711365280924616}
2026-08-05 23:15:55 | INFO     | pixcell.training.trainer.supervised:save_checkpoint:257 - Saved checkpoint -> checkpoints/best_model.pt
2026-08-05 23:15:55 | INFO     | pixcell.training.trainer.supervised:fit:315 - ========== Epoch 36 ==========
2026-08-05 23:15:55 | INFO     | pixcell.training.trainer.supervised:train_epoch:90 - Train epoch 36 | batches=161



Train 36:  67%|██████▋   | 108/161 [04:18<02:18,  2.61s/it, loss=0.921]